# Context-Aware and Retrieval-Augmented Sarcasm Detection in Social Media

## Notebook 4: Retrieval examples, LLM prompts, and final interpretation

This notebook adds the explanation layer. Retrieval shows similar labeled examples, and the prompt builders create zero-shot and few-shot examples without calling any paid API.


In [1]:
DATASET_SLUG = "/kaggle/input/datasets/danofer/sarcasm"
INPUT_DIR = "/kaggle/input/notebooks/minatahmasebi/"
WORKING_DIR = "/kaggle/working/"
OUTPUT_DIR = "/kaggle/working/"

KAGGLE_NOTEBOOK_INPUT_NAMES = [
    "01-data-and-tfidf-baseline",
    "02-context-free-transformer",
    "03-context-aware-transformer",
    "04-retrieval-and-llm-prompts",
    "01_data_and_tfidf_baseline",
    "02_context_free_transformer",
    "03_context_aware_transformer",
    "04_retrieval_and_llm_prompts",
]

MODEL_NAME = "distilroberta-base"
RANDOM_SEED = 42

DEBUG = False
SAMPLE_SIZE = 20000

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
GRADIENT_ACCUMULATION_STEPS = 1

MAX_TRANSFORMER_TRAIN_ROWS = None
MAX_TRANSFORMER_EVAL_ROWS = None
CPU_MAX_TRANSFORMER_TRAIN_ROWS = 2000
CPU_MAX_TRANSFORMER_EVAL_ROWS = 500
REQUIRE_GPU_FOR_FULL_TRANSFORMER = True

RUN_TFIDF = True
RUN_CONTEXT_FREE_TRANSFORMER = True
RUN_CONTEXT_AWARE_TRANSFORMER = True
RUN_RAG = True
RUN_LLM_PROMPTS = True

MAX_RETRIEVAL_TRAIN_ROWS = 50000


## Install retrieval dependencies

This simple cell installs the packages Kaggle may not have for retrieval. If FAISS is unavailable, the code falls back to scikit-learn nearest neighbors.


In [2]:
!pip install -q sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.9 MB/s eta 0:00:00


## Imports and GPU check

Sentence embeddings use a GPU if one is available. The fallback retrieval path works on CPU.


In [3]:
import json
import os
import random
import shutil
import textwrap
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Retrieval will use CPU.")


CUDA available: True
GPU: Tesla T4


## Shared helpers

Files are searched in `/kaggle/working/` and Kaggle notebook-output folders under `/kaggle/input/<notebook-name>/`.


In [4]:
def kaggle_notebook_input_roots():
    input_root = Path(INPUT_DIR)
    roots = []
    if not input_root.exists():
        return roots

    for notebook_name in KAGGLE_NOTEBOOK_INPUT_NAMES:
        direct_root = input_root / notebook_name
        if direct_root.exists() and direct_root.is_dir():
            roots.append(direct_root)

        for nested_root in input_root.rglob(notebook_name):
            if nested_root.exists() and nested_root.is_dir():
                roots.append(nested_root)

    unique_roots = []
    seen = set()
    for root in roots:
        resolved = root.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots


def prepare_input_search():
    print("Kaggle input search uses this order:")
    for root in search_roots():
        status = "exists" if root.exists() else "missing"
        print(f"- {root} [{status}]")
    print("Previous notebook outputs are expected at /kaggle/input/<notebook-name>/<output-files>.")


def search_roots():
    roots = [Path(WORKING_DIR)]
    roots.extend(kaggle_notebook_input_roots())
    roots.append(Path(INPUT_DIR))

    unique_roots = []
    seen = set()
    for root in roots:
        resolved = root.resolve() if root.exists() else root
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots


def find_file_recursive(filename):
    matches = []
    roots = search_roots()
    for root in roots:
        if root.exists():
            matches.extend(sorted(root.rglob(filename)))
    if not matches:
        return None

    def rank(path):
        resolved = path.resolve()
        for index, root in enumerate(roots):
            if not root.exists():
                continue
            root_resolved = root.resolve()
            if resolved == root_resolved or root_resolved in resolved.parents:
                return (index, len(str(resolved)))
        return (len(roots), len(str(resolved)))

    matches = sorted(set(matches), key=rank)
    return matches[0]


def require_file(filename, missing_message):
    path = find_file_recursive(filename)
    if path is None:
        print(missing_message)
        print("Searched these locations:")
        for root in search_roots():
            print(f"- {root}")
        raise FileNotFoundError(missing_message)
    print(f"Found {filename}: {path}")
    return path


def copy_to_working(path):
    path = Path(path)
    destination = Path(OUTPUT_DIR) / path.name
    if path.exists() and path.resolve() != destination.resolve():
        shutil.copy2(path, destination)
    return destination


def zip_outputs(zip_name, items):
    zip_path = Path(OUTPUT_DIR) / zip_name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        for item in items:
            path = Path(OUTPUT_DIR) / item
            if path.is_file():
                zipf.write(path, arcname=path.name)
            elif path.is_dir():
                for file_path in path.rglob("*"):
                    if file_path.is_file():
                        zipf.write(file_path, arcname=str(file_path.relative_to(Path(OUTPUT_DIR))))
            else:
                print(f"Skipping missing item: {path}")
    print(f"Created ZIP: {zip_path}")
    return zip_path


## Load prior outputs

`predictions_test.csv` comes from Notebook 3. If it is missing, run Notebook 3 or attach its output ZIP.


In [5]:
prepare_input_search()

train_path = require_file(
    "train_split.csv",
    "Missing train_split.csv. Please run Notebook 1 first or attach Notebook 1 outputs as a Kaggle dataset.",
)
validation_path = require_file(
    "validation_split.csv",
    "Missing validation_split.csv. Please run Notebook 1 first or attach Notebook 1 outputs as a Kaggle dataset.",
)
test_path = require_file(
    "test_split.csv",
    "Missing test_split.csv. Please run Notebook 1 first or attach Notebook 1 outputs as a Kaggle dataset.",
)
predictions_path = require_file(
    "predictions_test.csv",
    "Missing predictions_test.csv. Please run Notebook 3 first or attach Notebook 3 outputs as a Kaggle dataset.",
)
metrics_path = require_file(
    "metrics_comparison.csv",
    "Missing metrics_comparison.csv. Please run Notebook 3 first or attach Notebook 3 outputs as a Kaggle dataset.",
)
error_path = require_file(
    "error_analysis.csv",
    "Missing error_analysis.csv. Please run Notebook 3 first or attach Notebook 3 outputs as a Kaggle dataset.",
)

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)
predictions_test = pd.read_csv(predictions_path)
metrics_comparison = pd.read_csv(metrics_path)
error_analysis = pd.read_csv(error_path)

for source_path in [train_path, validation_path, test_path, predictions_path, metrics_path, error_path]:
    copy_to_working(source_path)

print("Train shape:", train_df.shape)
print("Test predictions shape:", predictions_test.shape)
display(metrics_comparison)
display(predictions_test.head(3))


Kaggle input search uses this order:
- /kaggle/working [exists]
- /kaggle/input/notebooks/minatahmasebi/01-data-and-tfidf-baseline [exists]
- /kaggle/input/notebooks/minatahmasebi/03-context-aware-transformer [exists]
- /kaggle/input/notebooks/minatahmasebi [exists]
Previous notebook outputs are expected at /kaggle/input/<notebook-name>/<output-files>.
Found train_split.csv: /kaggle/input/notebooks/minatahmasebi/01-data-and-tfidf-baseline/train_split.csv
Found validation_split.csv: /kaggle/input/notebooks/minatahmasebi/01-data-and-tfidf-baseline/validation_split.csv
Found test_split.csv: /kaggle/input/notebooks/minatahmasebi/01-data-and-tfidf-baseline/test_split.csv
Found predictions_test.csv: /kaggle/input/notebooks/minatahmasebi/03-context-aware-transformer/predictions_test.csv
Found metrics_comparison.csv: /kaggle/input/notebooks/minatahmasebi/01-data-and-tfidf-baseline/metrics_comparison.csv
Found error_analysis.csv: /kaggle/input/notebooks/minatahmasebi/03-context-aware-transforme

,model_name,input_type,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,weighted_f1,roc_auc,average_precision,test_samples,positive_samples,negative_samples,notes
0,TF-IDF + Logistic Regression,comment,0.722012,0.722018,0.739224,0.686163,0.711706,0.721657,0.721655,0.79479,0.801186,151496,75759,75737,Context-free TF-IDF baseline using comment onl...


,example_id,parent_comment,comment,true_label,tfidf_pred,tfidf_prob,context_free_pred,context_free_prob,context_aware_pred,context_aware_prob
0,544256,american OP gets shut down,murrica!,0,1,0.527993,1,0.691347,1,0.776398
1,129912,"10, The words ""Southern Strategy""",You are now banned from r/conservative,0,0,0.411966,1,0.772754,1,0.600449
2,507774,Space for two tails.,I like this kind of twin tails.,0,0,0.335020,0,0.097633,0,0.112222


## Retrieval sample

Retrieval is used for explanation, not final supervised training, so using a representative sample is acceptable and faster.


In [6]:
if len(train_df) > MAX_RETRIEVAL_TRAIN_ROWS:
    retrieval_train_df, _ = train_test_split(
        train_df,
        train_size=MAX_RETRIEVAL_TRAIN_ROWS,
        stratify=train_df["label"],
        random_state=RANDOM_SEED,
    )
    retrieval_train_df = retrieval_train_df.reset_index(drop=True)
    print(f"Using a stratified retrieval sample of {len(retrieval_train_df):,} training rows.")
else:
    retrieval_train_df = train_df.reset_index(drop=True).copy()
    print(f"Using all {len(retrieval_train_df):,} training rows for retrieval.")


Using a stratified retrieval sample of 50,000 training rows.


## Build the retrieval index

The preferred path uses `sentence-transformers/all-MiniLM-L6-v2` and FAISS. If either is unavailable, the notebook falls back to scikit-learn nearest neighbors.


In [7]:
from sklearn.model_selection import train_test_split

def make_retrieval_text(parent_comment, comment):
    return f"{str(parent_comment)} [REPLY] {str(comment)}"


train_retrieval_texts = [
    make_retrieval_text(parent, comment)
    for parent, comment in zip(retrieval_train_df["parent_comment"], retrieval_train_df["comment"])
]

embedding_backend = None
index_backend = None
sentence_model = None
vectorizer = None
train_embeddings = None
faiss_index = None
sklearn_index = None


def build_embeddings(texts):
    global embedding_backend, sentence_model, vectorizer
    try:
        from sentence_transformers import SentenceTransformer

        sentence_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
        embeddings = sentence_model.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embedding_backend = "sentence-transformers/all-MiniLM-L6-v2"
        return np.asarray(embeddings, dtype="float32")
    except Exception as exc:
        print("Sentence-transformers could not be used:", exc)
        print("Falling back to TF-IDF vectors for retrieval.")
        from sklearn.feature_extraction.text import TfidfVectorizer

        vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
        embedding_backend = "TF-IDF fallback"
        return vectorizer.fit_transform(texts)


def encode_query(texts):
    if embedding_backend == "sentence-transformers/all-MiniLM-L6-v2":
        return np.asarray(sentence_model.encode(texts, normalize_embeddings=True), dtype="float32")
    return vectorizer.transform(texts)


if RUN_RAG:
    train_embeddings = build_embeddings(train_retrieval_texts)

    if embedding_backend == "sentence-transformers/all-MiniLM-L6-v2":
        try:
            import faiss

            dimension = train_embeddings.shape[1]
            faiss_index = faiss.IndexFlatIP(dimension)
            faiss_index.add(train_embeddings)
            index_backend = "FAISS cosine/IP over normalized sentence embeddings"
        except Exception as exc:
            print("FAISS is unavailable, falling back to sklearn NearestNeighbors:", exc)
            from sklearn.neighbors import NearestNeighbors

            sklearn_index = NearestNeighbors(metric="cosine", algorithm="auto")
            sklearn_index.fit(train_embeddings)
            index_backend = "sklearn NearestNeighbors over sentence embeddings"
    else:
        from sklearn.neighbors import NearestNeighbors

        sklearn_index = NearestNeighbors(metric="cosine", algorithm="auto")
        sklearn_index.fit(train_embeddings)
        index_backend = "sklearn NearestNeighbors over TF-IDF vectors"

    print("Embedding backend:", embedding_backend)
    print("Index backend:", index_backend)
else:
    print("RUN_RAG is False, so retrieval was skipped.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Embedding backend: sentence-transformers/all-MiniLM-L6-v2
Index backend: FAISS cosine/IP over normalized sentence embeddings


## Retrieve similar examples

The retrieval function returns labeled training examples close to a test conversation.


In [8]:
def retrieve_similar_examples(parent_comment, comment, k=5):
    if train_embeddings is None:
        raise RuntimeError("Retrieval index has not been built. Set RUN_RAG=True and run the previous cell.")

    query_text = make_retrieval_text(parent_comment, comment)
    query_embedding = encode_query([query_text])

    if faiss_index is not None:
        scores, indices = faiss_index.search(query_embedding.astype("float32"), k)
        scores = scores[0]
        indices = indices[0]
    else:
        distances, indices = sklearn_index.kneighbors(query_embedding, n_neighbors=k)
        distances = distances[0]
        indices = indices[0]
        scores = 1 - distances

    rows = retrieval_train_df.iloc[indices][["parent_comment", "comment", "label"]].copy()
    rows["similarity_score"] = scores
    return rows.reset_index(drop=True)


retrieval_frames = []
if RUN_RAG:
    demo_examples = predictions_test.head(5).copy()
    for query_number, (_, row) in enumerate(demo_examples.iterrows(), start=1):
        retrieved = retrieve_similar_examples(row["parent_comment"], row["comment"], k=5)
        retrieved.insert(0, "query_number", query_number)
        retrieved.insert(1, "query_parent_comment", row["parent_comment"])
        retrieved.insert(2, "query_comment", row["comment"])
        retrieved.insert(3, "query_true_label", row["true_label"])
        retrieved.insert(4, "query_context_aware_pred", row.get("context_aware_pred", np.nan))
        retrieval_frames.append(retrieved)

        print("\nQuery", query_number)
        print("Parent:", str(row["parent_comment"])[:300])
        print("Reply:", str(row["comment"])[:300])
        display(retrieved[["parent_comment", "comment", "label", "similarity_score"]])

    retrieval_examples = pd.concat(retrieval_frames, ignore_index=True)
else:
    retrieval_examples = pd.DataFrame(
        columns=[
            "query_number",
            "query_parent_comment",
            "query_comment",
            "query_true_label",
            "query_context_aware_pred",
            "parent_comment",
            "comment",
            "label",
            "similarity_score",
        ]
    )

retrieval_examples.to_csv(Path(OUTPUT_DIR) / "retrieval_examples.csv", index=False)
print("Saved retrieval_examples.csv")



Query 1
Parent: american OP gets shut down
Reply: murrica!


,parent_comment,comment,label,similarity_score
0,But that is British pronunciation,"Yeah well this is 'Murica, speak 'Murican!",1,0.506798
1,####I don't speak yo damn language boi.,#### I speak 'murican.....,0,0.485265
2,'Murica!,Why is e'erbody laughin?,1,0.479461
3,"Heads up, America",pervs...,1,0.467642
4,Cause OP is American. They love to do this. Go...,I'm not.,0,0.464772



Query 2
Parent: 10, The words "Southern Strategy"
Reply: You are now banned from r/conservative


,parent_comment,comment,label,similarity_score
0,"Not a Tea Party, a Confederate Party","""The South will rise again"" and all that.",0,0.510814
1,Conservative here.,"Thank you, ""Conservative"" it is!",0,0.496143
2,America's youth. They're not out of control.,Hey you dropped this.,1,0.461165
3,Well you sure deleted your first comment fast ...,The comments are being delete by the mods.,0,0.456100
4,"Making conservatives look bad, not all conserv...",/#notallconservatives,1,0.455828



Query 3
Parent: Space for two tails.
Reply: I like this kind of twin tails.


...


Query 4
Parent: I can't imagine it would be too terribly difficult to include anyway. That said, if this happens to be the hang up then rewriting the game to not use the roms and instead manually scripting it may be the future of PokeMMO.
Reply: Implementing johto via liquid crystal wouldn't be too terribly hard, i imagine.


,parent_comment,comment,label,similarity_score
0,OSBuddy has been in the game for so long it's ...,Yeah same with NMZ and splashing.,1,0.384035
1,"WHAT TO DOOOO?!?!? When your mobo is fried, sh...","There's this new game called ""Outside"" it's re...",1,0.376806
2,Hey those pentas were cool and all but is no o...,If you had the chance for a penta before endin...,0,0.373028
3,Smiteguru has an aproximation of your real elo...,Everyone knows it does but why wouldn't they j...,0,0.372598
4,Ideally it should be handled how Rocket League...,"As said before, we also need a wheel just to c...",1,0.371653



Query 5
Parent: My favorite Lucille Bluth face (S1E12), what's yours?
Reply: This is the reason why Jessica Walter is my favorite character on the show.


,parent_comment,comment,label,similarity_score
0,lucy liu. her face is just not very attractive,racist,1,0.492097
1,ScarJo. I actually really like her voice acting.,insert lenny face,0,0.453379
2,My personal favorite face swap.,What face swap?,1,0.446055
3,My name is Justin and I thought it was the coo...,I found that it was such a fitting name for th...,0,0.441075
4,Sister of the Year,Someone replace her face with Luigi's stare,0,0.438478


Saved retrieval_examples.csv


## LLM prompt builders

No paid API is called. These functions only write prompts for optional zero-shot and few-shot comparison.


In [9]:
def build_zero_shot_prompt(parent_comment, comment):
    return textwrap.dedent(
        f'''
        You are judging sarcasm in a Reddit conversation.

        Parent comment:
        {parent_comment}

        Target reply:
        {comment}

        Is the target reply sarcastic? Answer 0 or 1 and explain briefly.
        Use 1 for sarcastic and 0 for non-sarcastic.
        '''
    ).strip()


def build_few_shot_prompt(parent_comment, comment, retrieved_examples):
    example_blocks = []
    for i, (_, example) in enumerate(retrieved_examples.head(3).iterrows(), start=1):
        example_blocks.append(
            textwrap.dedent(
                f'''
                Example {i}
                Parent comment: {example["parent_comment"]}
                Target reply: {example["comment"]}
                Label: {int(example["label"])}
                '''
            ).strip()
        )

    examples_text = "\n\n".join(example_blocks) if example_blocks else "No retrieved examples are available."
    return textwrap.dedent(
        f'''
        You are judging sarcasm in a Reddit conversation.

        Here are similar labeled examples:

        {examples_text}

        Now classify this new case.

        Parent comment:
        {parent_comment}

        Target reply:
        {comment}

        Is the target reply sarcastic? Answer 0 or 1 and explain briefly.
        Use 1 for sarcastic and 0 for non-sarcastic.
        '''
    ).strip()


## Save prompt examples

The text file is easy to read, and the CSV is easier to analyze later.


In [10]:
prompt_rows = []
prompt_text_blocks = []

if RUN_LLM_PROMPTS:
    prompt_examples = predictions_test.head(5).copy()
    for query_number, (_, row) in enumerate(prompt_examples.iterrows(), start=1):
        if RUN_RAG and train_embeddings is not None:
            retrieved = retrieve_similar_examples(row["parent_comment"], row["comment"], k=5)
        else:
            retrieved = pd.DataFrame(columns=["parent_comment", "comment", "label", "similarity_score"])

        zero_prompt = build_zero_shot_prompt(row["parent_comment"], row["comment"])
        few_prompt = build_few_shot_prompt(row["parent_comment"], row["comment"], retrieved)

        prompt_rows.append(
            {
                "query_number": query_number,
                "parent_comment": row["parent_comment"],
                "comment": row["comment"],
                "true_label": row["true_label"],
                "zero_shot_prompt": zero_prompt,
                "few_shot_prompt": few_prompt,
            }
        )

        prompt_text_blocks.append(
            f"==================== Query {query_number}: zero-shot ====================\n"
            + zero_prompt
            + "\n\n"
            + f"==================== Query {query_number}: few-shot ====================\n"
            + few_prompt
        )

    llm_prompt_examples = pd.DataFrame(prompt_rows)
else:
    llm_prompt_examples = pd.DataFrame(
        columns=[
            "query_number",
            "parent_comment",
            "comment",
            "true_label",
            "zero_shot_prompt",
            "few_shot_prompt",
        ]
    )

llm_prompt_examples.to_csv(Path(OUTPUT_DIR) / "llm_prompt_examples.csv", index=False)
with open(Path(OUTPUT_DIR) / "llm_prompt_examples.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(prompt_text_blocks))

print("Saved llm_prompt_examples.csv and llm_prompt_examples.txt")
display(llm_prompt_examples.head(2))


Saved llm_prompt_examples.csv and llm_prompt_examples.txt


,query_number,parent_comment,comment,true_label,zero_shot_prompt,few_shot_prompt
0,1,american OP gets shut down,murrica!,0,You are judging sarcasm in a Reddit conversati...,You are judging sarcasm in a Reddit conversati...
1,2,"10, The words ""Southern Strategy""",You are now banned from r/conservative,0,You are judging sarcasm in a Reddit conversati...,You are judging sarcasm in a Reddit conversati...


## Final conclusion

This conclusion is generated from `metrics_comparison.csv`, and it does not hide a weak result.


In [11]:
def find_model_row(metrics_df, phrase):
    mask = metrics_df["model_name"].astype(str).str.lower().str.contains(phrase.lower(), regex=False)
    if mask.any():
        return metrics_df.loc[mask].iloc[0]
    return None


def print_best(metrics_df, metric, label):
    if metric not in metrics_df.columns or metrics_df[metric].dropna().empty:
        print(f"Best {label}: not available")
        return None
    row = metrics_df.dropna(subset=[metric]).sort_values(metric, ascending=False).iloc[0]
    print(f"Best {label}: {row['model_name']} with {metric} = {row[metric]:.4f}")
    return row


usable_metrics = metrics_comparison.copy()
print("Final project conclusion")
print("========================")
print_best(usable_metrics, "accuracy", "accuracy")
print_best(usable_metrics, "balanced_accuracy", "balanced accuracy")
print_best(usable_metrics, "precision", "precision")
print_best(usable_metrics, "recall", "recall")
print_best(usable_metrics, "f1", "F1")
print_best(usable_metrics, "macro_f1", "macro-F1")
print_best(usable_metrics, "weighted_f1", "weighted-F1")
print_best(usable_metrics, "roc_auc", "ROC-AUC")
print_best(usable_metrics, "average_precision", "average precision")

context_free = find_model_row(usable_metrics, "context-free transformer")
context_aware = find_model_row(usable_metrics, "context-aware transformer")

if context_free is not None and context_aware is not None:
    f1_difference = context_aware["f1"] - context_free["f1"]
    macro_f1_difference = context_aware["macro_f1"] - context_free["macro_f1"]
    print(f"Context-aware minus context-free F1: {f1_difference:+.4f}")
    print(f"Context-aware minus context-free macro-F1: {macro_f1_difference:+.4f}")

    if f1_difference > 0:
        print("The context-aware transformer beats the context-free transformer by F1 in this run.")
    else:
        print("The context-aware transformer does not beat the context-free transformer by F1 in this run.")
        print(
            "A fair explanation is that context may be noisy, target comments may already contain sarcasm markers, "
            "max_length may truncate useful context, or the training/sample size may be limited."
        )
else:
    print("The context-free and context-aware transformer rows are not both available, so their direct comparison cannot be computed.")

if Path(OUTPUT_DIR, "retrieval_examples.csv").exists():
    print("Retrieval examples are useful for explanation, but they are not the final supervised training method.")


Final project conclusion
Best accuracy: TF-IDF + Logistic Regression with accuracy = 0.7220
Best balanced accuracy: TF-IDF + Logistic Regression with balanced_accuracy = 0.7220
Best precision: TF-IDF + Logistic Regression with precision = 0.7392
Best recall: TF-IDF + Logistic Regression with recall = 0.6862
Best F1: TF-IDF + Logistic Regression with f1 = 0.7117
Best macro-F1: TF-IDF + Logistic Regression with macro_f1 = 0.7217
Best weighted-F1: TF-IDF + Logistic Regression with weighted_f1 = 0.7217
Best ROC-AUC: TF-IDF + Logistic Regression with roc_auc = 0.7948
Best average precision: TF-IDF + Logistic Regression with average_precision = 0.8012
The context-free and context-aware transformer rows are not both available, so their direct comparison cannot be computed.
Retrieval examples are useful for explanation, but they are not the final supervised training method.


## Save final outputs

This ZIP contains the retrieval examples, prompt examples, and final comparison files.


In [12]:
final_outputs = [
    "retrieval_examples.csv",
    "llm_prompt_examples.csv",
    "llm_prompt_examples.txt",
    "predictions_test.csv",
    "metrics_comparison.csv",
    "error_analysis.csv",
]

zip_outputs("sarcasm_final_outputs.zip", final_outputs)
print("Notebook 4 complete.")


Created ZIP: /kaggle/working/sarcasm_final_outputs.zip
Notebook 4 complete.
